<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/Copy_of_MNIST_PEPITA_Repository_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST PEPITA runs using the supplied repository

This notebook runs the supplied PEPITA project code on MNIST for the two one-hidden-layer settings reported in the paper: without weight decay and with weight decay \(10^{-5}\). The settings run one after another and use the same seed (0) as the existing independent MNIST notebook, making a seed-matched implementation comparison possible.

The existing MNIST notebook remains useful as a supplementary implementation. These new runs are the repository-based MNIST results. The repository's MNIST loader converts pixels to \([0,1]\) without MNIST mean/std normalization when normalization is disabled. Do not silently add another preprocessing change; record this difference when interpreting against the paper and the independent notebook.

The repository evaluates its test loader as Lightning's validation loader each epoch. Use the final epoch only; do not choose a checkpoint or tune settings based on those test metrics.


## 1. Select and verify a GPU
In Colab select Runtime → Change runtime type → T4 GPU, then run this cell.

In [ ]:
from pathlib import Path
import os, sys, platform, subprocess, textwrap, json, time, shutil, zipfile

print('Colab Python:', sys.version.split()[0])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'], text=True, capture_output=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No GPU detected. Select a GPU runtime and rerun.')
print('GPU:', gpu.stdout.strip())
!nvidia-smi


## 2. Upload the supplied PEPITA project ZIP
Upload the same complete source ZIP used for the CIFAR-100 Colab run. It must contain exactly one `scripts/train.py`, the `pepita/` source directory, and the project metadata. Do not upload a run-results ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one ZIP containing the complete PEPITA project source.')
zip_name = next(iter(uploaded))
if not zip_name.lower().endswith('.zip'):
    raise ValueError('Please upload the PEPITA project source ZIP.')

extract_root = Path('/content/pepita_upload')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_root)
train_scripts = list(extract_root.rglob('scripts/train.py'))
if len(train_scripts) != 1:
    raise RuntimeError(f'Expected exactly one scripts/train.py; found {len(train_scripts)}.')
PROJECT = train_scripts[0].parent.parent.resolve()
if not (PROJECT / 'pepita' / 'core' / 'trainer.py').exists():
    raise RuntimeError('ZIP is missing pepita/core/trainer.py; upload the complete project source.')
os.chdir(PROJECT)
print('Project folder:', PROJECT)


## 3. Create the pinned research environment
This matches the CUDA/PyTorch environment used for the completed CIFAR-100 run.

In [ ]:
micromamba_root = '/content/micromamba'
os.environ['MAMBA_ROOT_PREFIX'] = micromamba_root
if not Path('/content/bin/micromamba').exists():
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba", shell=True, check=True)
MM = '/content/bin/micromamba'
subprocess.run([MM, 'create', '-y', '-n', 'pepita', 'python=3.10', 'pip'], check=True)
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-m', 'pip', 'install', '--upgrade', 'pip==24.0'], check=True)
packages = [
    'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117',
    'pytorch-lightning==1.6.5', 'torchmetrics==0.9.2',
    'numpy==1.21.2', 'Pillow==8.4.0', 'scikit-learn==1.2.1', 'scipy==1.10.1',
    'loguru==0.6.0', 'setuptools==80.9.0', 'ray[tune]==2.3.1', 'yacs==0.1.8',
    'tensorboard==2.12.3', 'protobuf<4'
]
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-m', 'pip', 'install',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu117', *packages], check=True)
print('Pinned environment installation complete.')


## 4. Apply the guarded zero-validation-split fix and check GPU compatibility
The supplied data loader creates a null validation dataset when `val_split=0`, then attempts to wrap it in a DataLoader. This small guard preserves all training examples and lets the repository's existing test-loader validation hook run. The patch stops if the source differs from the reviewed file pattern.

In [ ]:
patch_script = r'''from pathlib import Path
p = Path('pepita/dataset/getdataloader.py')
s = p.read_text(encoding='utf-8')
old = '    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True)\n'
new = '    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True) if val_data is not None else None\n'
count = s.count(old)
if count != 3:
    raise RuntimeError(f'Expected three validation-loader lines, found {count}; source differs, so no patch was applied.')
p.write_text(s.replace(old, new), encoding='utf-8')
print('Applied guarded val_split=0 data-loader fix.')
'''
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-c', patch_script], cwd=PROJECT, check=True)
check_script = "import torch, pytorch_lightning as pl; cap=torch.cuda.get_device_capability(0); print('PyTorch',torch.__version__,'Lightning',pl.__version__,'GPU',torch.cuda.get_device_name(0),'capability',cap,'CUDA',torch.version.cuda); arch=torch.cuda.get_arch_list(); required=f'sm_{cap[0]}{cap[1]}'; assert required in arch, f'{required} not supported by this PyTorch build: {arch}'"
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-c', check_script], check=True)


## 5. Set the paper's one-hidden-layer MNIST presets
Start with seed 0 to match the seed used by the independent notebook. For a multi-seed study, rerun later with a predeclared seed list (for example, `list(range(10))`) and report every run plus mean and standard deviation.

In [ ]:
SEEDS = [0]
PRESETS = [
    {'name': 'mnist_repo_pepita_1hidden_noWD', 'weight_decay': 0.0},
    {'name': 'mnist_repo_pepita_1hidden_WD1e-5', 'weight_decay': 1e-5},
]
COMMON = [
    '-d', 'mnist', '-ls', '1024', '-lr', '0.1', '-e', '100', '-bs', '64',
    '-mom', '0.9', '-dr', '0.1', '-lrd', '0.1', '-de', '60', '90',
    '-li', 'he_uniform', '-bi', 'uniform', '-bstd', '0.05',
    '-w', '2', '-wmlr', '0.1', '-wmwd', '0.5',
]
print('Seeds:', SEEDS)
print('Presets:', PRESETS)
print('Repository preprocessing: ToTensor only (no MNIST mean/std normalization).')


## 6. Run the two MNIST experiments sequentially
Allow the cell to finish. Colab may disconnect if the runtime is idle or reaches its session limit; download the result archive promptly when the final cell offers it.

In [ ]:
from datetime import datetime, timezone

run_records = []
for preset in PRESETS:
    for seed in SEEDS:
        experiment = f"{preset['name']}_seed{seed}"
        cmd = [MM, 'run', '-n', 'pepita', 'python', 'scripts/train.py',
               '-en', experiment, *COMMON, '-wd', str(preset['weight_decay']), '-s', str(seed)]
        run_started_utc = datetime.now(timezone.utc).isoformat()
        print('\n=== Starting', experiment, '===')
        print('Started UTC:', run_started_utc)
        print('Command:', ' '.join(cmd))
        log_path = PROJECT / 'experiments' / experiment / 'colab_console.log'
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, 'w', encoding='utf-8') as log:
            proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
            code = proc.wait()
        run_finished_utc = datetime.now(timezone.utc).isoformat()
        if code != 0:
            raise RuntimeError(f'{experiment} exited with status {code}; inspect {log_path}')
        info_cmd = "import torch,pytorch_lightning as pl,platform,json; print(json.dumps({'python':platform.python_version(),'pytorch':torch.__version__,'torchvision':__import__('torchvision').__version__,'pytorch_lightning':pl.__version__,'cuda_runtime':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'gpu_capability':torch.cuda.get_device_capability(0)}))"
        runtime = json.loads(subprocess.check_output([MM, 'run', '-n', 'pepita', 'python', '-c', info_cmd], cwd=PROJECT, text=True))
        run_dir = PROJECT / 'experiments' / experiment
        run_meta = {
            'experiment': experiment, 'dataset': 'MNIST', 'method': 'PEPITA, one hidden layer',
            'seed': seed, 'weight_decay': preset['weight_decay'], 'learning_rate': 0.1,
            'hidden_units': 1024, 'batch_size': 64, 'epochs': 100,
            'lr_milestones': [60, 90], 'lr_gamma': 0.1, 'dropout': 0.1,
            'augmentation': False, 'normalization': 'ToTensor only; no mean/std normalization',
            'runtime': runtime, 'started_utc': run_started_utc, 'finished_utc': run_finished_utc,
            'evaluation_note': 'val_acc and val_loss are computed on the test loader each epoch; final epoch is reported, no best-test-epoch selection.'
        }
        (run_dir / 'runtime_info.json').write_text(json.dumps(run_meta, indent=2), encoding='utf-8')
        run_records.append(run_meta)
        print('Finished UTC:', run_finished_utc)
        print('Completed:', experiment)
print('\nBoth requested MNIST presets completed.')


## 7. Review final metrics from the TensorBoard files
The repository logs final epoch scalars as `val_acc` and `val_loss`. Lightning's scalar step is zero-based, so step 99 corresponds to epoch 100.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

for record in run_records:
    exp_dir = PROJECT / 'experiments' / record['experiment']
    event_files = list(exp_dir.rglob('events.out.tfevents.*'))
    main_files = [p for p in event_files if 'angle_al_total' not in str(p) and 'weight_norms' not in str(p) and 'b_means' not in str(p) and 'b_stds' not in str(p)]
    if not main_files:
        print(record['experiment'], ': no main event file found')
        continue
    acc = EventAccumulator(str(main_files[0]), size_guidance={'scalars': 0})
    acc.Reload()
    result = {'experiment': record['experiment'], 'seed': record['seed'], 'weight_decay': record['weight_decay']}
    for tag in ('val_acc', 'val_loss'):
        if tag not in acc.Tags().get('scalars', []):
            print('Available tags:', acc.Tags().get('scalars', []))
            raise KeyError(f'Missing expected scalar {tag} in {record["experiment"]}')
        ev = max(acc.Scalars(tag), key=lambda item: item.step)
        result[tag] = float(ev.value)
        result[tag + '_step'] = int(ev.step)
    print(json.dumps(result, indent=2))
    record['final_logged_accuracy'] = result['val_acc']
    record['final_logged_loss'] = result['val_loss']


## 8. Save the full run evidence
The archive contains both experiment folders with config files, logs, TensorBoard events, and runtime metadata. Keep it with the downloaded CIFAR run archive and do not edit the archived source evidence.

In [ ]:
from google.colab import files

archive_path = Path('/content/mnist_repo_pepita_runs.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for record in run_records:
        exp_dir = PROJECT / 'experiments' / record['experiment']
        for file in exp_dir.rglob('*'):
            if file.is_file():
                zf.write(file, Path('experiments') / record['experiment'] / file.relative_to(exp_dir))
summary_path = Path('/content/mnist_repo_pepita_metrics.json')
summary_path.write_text(json.dumps(run_records, indent=2), encoding='utf-8')
print('Evidence archive:', archive_path)
print('Metric summary:', summary_path)
files.download(str(archive_path))
files.download(str(summary_path))
